# Gene heatmaps: every isoform and ORF of chosen genes

One objective: for the genes you name, show every transcript isoform and every ORF of the gene across the samples, so you can see which isoforms differ between the groups. The whole-transcriptome heatmaps are in `heatmaps.ipynb`.

In [ ]:
source("src/heatmap.R")
source("src/paths.R")
loc <- run_locations()        # every folder of the run, from paths.json
comparison <- loc$comparison
genes <- c("HLA-B", "ERAP1", "ERAP2")   # genes to draw
exclude <- c("420_20190402")             # samples to leave out in the second drawing; none of them in this run is fine
groups <- read_groups(file.path(loc$leafcutter, paste0(comparison, ".lr_leafcutter.groups_file.txt")))
exclude <- intersect(exclude, groups$sample)
groups

**Explanation.** `genes` is the list to draw; change it to look at others. `exclude` names samples to leave out in the second drawing of each gene. A sample that is not in this run is ignored, so the same list can be used for every comparison.

In [ ]:
tx_file  <- file.path(loc$de_transcript, paste0(comparison, "_DTE_edgeR_normalized_CPM_matrix.txt"))
orf_file <- file.path(loc$de_orf, paste0(comparison, "_DE_ORF_edgeR_normalized_CPM_matrix.txt"))
tx  <- read_expression(tx_file, groups, id_col = "isoform_id", label_col = "transcript_name")
orf <- read_expression(orf_file, groups, id_col = "orf_all_isoform_id", label_col = "orf_base_id")
tx_gene  <- table_column(tx_file, "gene_name")
orf_gene <- sub("::.*$", "", table_column(orf_file, "orf_base_id"))
rbind(isoforms = tapply(rep(1, length(tx_gene)), factor(tx_gene, genes), sum),
      orfs     = tapply(rep(1, length(orf_gene)), factor(orf_gene, genes), sum))

**Explanation.** The expression of every isoform and ORF is read as in `heatmaps.ipynb`. Each isoform's gene is its `gene_name` in the table; each ORF's gene is the part of its name before `::`. The small table counts how many isoforms (first row) and ORFs (second row) each chosen gene has in this run; `NA` means the gene has none.

In [ ]:
for (g in genes) {
  m <- gene_rows(tx, tx_gene, g)
  if (nrow(m) > 0) show_heatmap(expression_heatmap(m, groups, title = paste0(g, " transcript isoforms, all samples"), mean_bar = TRUE), paste0("gene_", g, "_isoforms"))
  o <- gene_rows(orf, orf_gene, g)
  if (nrow(o) > 0) show_heatmap(expression_heatmap(o, groups, title = paste0(g, " ORFs, all samples"), mean_bar = TRUE), paste0("gene_", g, "_orfs"))
}

**Explanation.** For each gene, one heatmap of its transcript isoforms and one of its ORFs. Each row is one isoform or ORF, each column a sample, and the columns are split by group. Colour is the row's log2(CPM + 1), centred and scaled, so red means the sample is above that row's average. The bars at the right show each row's average CPM, because the colours alone hide which isoforms are common and which are rare. A row that is high in one group and low in the other is an isoform that differs between the groups. With only a few samples per group, one sample can make a row look different, so check the bar and the sample columns before reading much into a single row.

In [ ]:
if (length(exclude) > 0) {
  g2 <- drop_samples(groups, exclude)
  label <- paste(exclude, collapse = ", ")
  for (g in genes) {
    m <- gene_rows(tx, tx_gene, g)
    if (nrow(m) > 0) show_heatmap(expression_heatmap(m[, g2$sample, drop = FALSE], g2, title = paste0(g, " transcript isoforms, without ", label), mean_bar = TRUE), paste0("gene_", g, "_isoforms_without"))
    o <- gene_rows(orf, orf_gene, g)
    if (nrow(o) > 0) show_heatmap(expression_heatmap(o[, g2$sample, drop = FALSE], g2, title = paste0(g, " ORFs, without ", label), mean_bar = TRUE), paste0("gene_", g, "_orfs_without"))
  }
} else {
  message("no sample to leave out: no second drawing")
}

**Explanation.** The same heatmaps without the samples in `exclude`, when any of them are in this run. Compare each with its all-samples drawing: if the pattern holds, one sample was not driving it.